# Lesson 9 — Principal Component Analysis (PCA)

## 1. Why should I care?

In Remote Sensing, one satellite image can contain **many correlated features**: spectral bands, NDVI, NDWI, NDBI, LST, elevation, etc.

PCA helps us **compress many related variables into a smaller number of new variables while keeping as much information as possible**.

This is especially useful when:

- You have **many features**.
- Features are strongly correlated.
- You want to visualize high-dimensional data.
- You want to reduce computational complexity before ML.

---

# 2. Core Concept

### The Big Idea

Suppose you have 10 features:

```text
Band 2
Band 3
Band 4
Band 5
Band 6
Band 7
NDVI
NDWI
NDBI
LST
````

Some of these contain overlapping information.

For example:

```text
Band 5 ─────┐
Band 6 ─────┼── similar vegetation information
NDVI  ──────┘
```

Instead of keeping all 10 separately, PCA creates **new variables called Principal Components (PCs)**.

```text
Original features

10 features
     ↓
    PCA
     ↓
PC1
PC2
PC3
...
```

Perhaps the first 3 components contain most of the useful variation:

```text
10 features → 3 components
```

So PCA is a form of **dimensionality reduction**.

---

# 3. Simple Everyday Example

Imagine describing 100 people using:

* Height
* Weight
* Waist size
* Shoe size
* Arm length
* Leg length

Many measurements are related.

PCA might discover that several measurements mainly represent a hidden dimension like:

> **Body size**

Another component might represent:

> **Body proportions**

So instead of six measurements, we might describe people reasonably well using just two components.

**Important:** PCA does not know what "body size" means.

It mathematically finds directions that capture the most variation.

---

# 4. Remote Sensing Example 🌍

Imagine you have Sentinel-2 pixels with:

```text
B2
B3
B4
B5
B6
B7
B8
B11
B12
NDVI
NDWI
```

That's **11 features per pixel**.

Some bands and indices are correlated.

For example:

```text
Vegetated pixel

High B8
High B5
High NDVI
```

These variables are partly describing the same underlying vegetation signal.

PCA transforms them into:

```text
Original data
11 features
     ↓
    PCA
     ↓
PC1
PC2
PC3
```

You can then create a map of PC1.

PC1 may capture the **largest source of variation** in the image.

PC2 captures the largest remaining variation.

PC3 captures the next largest remaining variation.

But remember:

> **PC1 does not automatically mean vegetation.**

You have to examine the component loadings and the spatial pattern to interpret it.

---

# 5. How PCA Works — Intuition

You already learned **vectors, matrices, eigenvectors and eigenvalues**, so we can connect PCA to that.

Imagine your data points look like this:

```text
        •
      •
    •
  •
•
```

The data has a strong direction from bottom-left to top-right.

PCA finds that direction.

### Principal Component 1

The direction containing the **largest amount of variation**.

```text
        •
      •
    •
  •
•  ───────────────→ PC1
```

### Principal Component 2

A direction perpendicular to PC1 that captures the largest remaining variation.

```text
          PC2
           ↑
           |
           |
───────────┼──────────→ PC1
           |
           |
```

Therefore:

**PC1 = most variation**

**PC2 = second-most variation**

**PC3 = third-most variation**

and so on.

---

# 6. What Does "Variance" Mean Here?

You already learned variance.

Variance tells us roughly:

> **How spread out the data is.**

PCA looks for directions where the data varies the most.

Imagine:

```text
Direction A:

•
 •
  •
   •
    •

Large variation
```

versus:

```text
Direction B:

  •••••
  •••••
  •••••

Small variation
```

PCA prioritizes Direction A.

That's why:

> **PCA tries to preserve the directions containing the most variance.**

---

# 7. PCA and Eigenvectors

This is where your linear algebra becomes useful.

PCA uses the **covariance structure** of the data.

Very roughly:

```text
Data
 ↓
Center / usually scale
 ↓
Covariance structure
 ↓
Eigenvectors + eigenvalues
 ↓
Principal Components
```

### Eigenvectors

The eigenvectors give us the **directions** of the principal components.

### Eigenvalues

The eigenvalues tell us **how much variance each direction explains**.

For example:

| Component | Explained Variance |
| --------- | -----------------: |
| PC1       |                55% |
| PC2       |                25% |
| PC3       |                10% |
| PC4       |                 5% |
| PC5       |                 3% |
| PC6       |                 2% |

Then:

```text
PC1 + PC2 = 80%
PC1 + PC2 + PC3 = 90%
```

So perhaps we can reduce:

```text
6 features → 3 PCs
```

while retaining about **90% of the variance**.

---

# 8. Standardization Is Important

This is particularly important for you.

Suppose your features are:

```text
Elevation:      0–3000 m
NDVI:          -1–1
Temperature:    10–40 °C
```

Elevation has much larger numerical values.

If we directly apply PCA, the scale of the variables can strongly affect the result.

Therefore, PCA is commonly applied after standardization:

```text
X_scaled = (X - mean) / standard deviation
```

After scaling:

```text
mean ≈ 0
standard deviation ≈ 1
```

In Python:

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
```

Then:

```python
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
```

Now your original data:

```text
many features
```

becomes:

```text
PC1   PC2
 ↓     ↓
•     •
•     •
•     •
```

---

# 9. Mini Python Example

Let's use the Iris dataset again.

```python
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
```

Import the dataset, scaling tool, and PCA.

```python
iris = load_iris()
X = iris.data
```

Get the four original features:

```text
sepal length
sepal width
petal length
petal width
```

Now scale them:

```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
```

Apply PCA:

```python
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
```

Now:

```text
4 features
   ↓
 PCA
   ↓
2 features
```

Check how much variance was retained:

```python
print(pca.explained_variance_ratio_)
```

You might get approximately:

```text
[0.73, 0.23]
```

Meaning:

```text
PC1 → ~73%
PC2 → ~23%
```

Together:

```text
~96%
```

So two components retain approximately 96% of the original variance.

---

# 10. PCA vs K-Means

This is an important distinction because you just learned K-Means.

### K-Means

Answers:

> **Can I divide my observations into groups?**

```text
Data → K-Means → clusters
```

### PCA

Answers:

> **Can I represent my data using fewer dimensions while preserving variation?**

```text
Data → PCA → fewer dimensions
```

They can also be combined:

```text
Satellite data
      ↓
Standardization
      ↓
PCA
      ↓
Reduced features
      ↓
K-Means
      ↓
Clusters
```

This can be useful when you have many highly correlated remote-sensing features.

---

# 11. When Should You Use PCA?

### Good Situations

Use PCA when:

* You have **many numerical features**.
* Features are strongly correlated.
* You want dimensionality reduction.
* You want to visualize high-dimensional data in 2D or 3D.
* You want to reduce redundancy.
* You want to preprocess data before another ML algorithm.

### Example

```text
50 satellite features
        ↓
       PCA
        ↓
10 components
        ↓
Random Forest / SVM / K-Means
```

---

# 12. When Should You NOT Use PCA?

PCA isn't automatically beneficial.

Avoid or carefully consider PCA when:

### 1. Interpretability is very important

Original:

```text
NDVI
NDWI
LST
B8
B11
```

is easy to understand.

But:

```text
PC1
PC2
PC3
```

is much harder to interpret.

### 2. You have only a few useful features

If you have:

```text
NDVI
LST
```

there may be little reason to use PCA.

### 3. Your variables aren't suitable

PCA is fundamentally designed around numerical variables and linear relationships.

### 4. You need the original physical meaning

For environmental research, sometimes the individual variables themselves are scientifically important.

---

# 13. Advantages

* Reduces dimensionality.
* Removes some redundancy caused by correlated variables.
* Can make visualization easier.
* Can reduce computational cost.
* Useful for high-dimensional remote-sensing datasets.
* Can help some ML models work more efficiently.

---

# 14. Limitations

* Principal components can be difficult to interpret.
* You must usually scale variables carefully.
* PCA is a **linear** transformation.
* Important information can sometimes be discarded when reducing dimensions.
* PCA doesn't know which information is scientifically important.
* Results depend on the variables you provide.

---

# 15. How Do We Evaluate PCA?

One of the most important measures is:

### Explained Variance Ratio

It tells us how much variation each component captures.

Example:

```text
PC1 → 60%
PC2 → 20%
PC3 → 10%
PC4 → 5%
PC5 → 3%
PC6 → 2%
```

Cumulative:

```text
PC1                 60%
PC1 + PC2           80%
PC1 + PC2 + PC3     90%
```

We might choose the first 3 components if approximately 90% is sufficient for our purpose.

---

# 16. The Big Picture

You should now see the relationship between several algorithms you've learned:

```text
                    MACHINE LEARNING
                           │
             ┌─────────────┴─────────────┐
             │                           │
        Supervised                  Unsupervised
             │                           │
     ┌───────┴───────┐             ┌─────┴─────┐
     │               │             │           │
Classification   Regression      K-Means      PCA
     │                               │           │
     ↓                               ↓           ↓
Predict label                    Find groups   Reduce
Predict value                                 dimensions
```

PCA isn't primarily a prediction algorithm.

It is a **dimensionality-reduction technique**.

---

# 17. Practice Together — Question 1

Imagine you have **100,000 Sentinel-2 pixels**.

For every pixel you calculate:

```text
B2
B3
B4
B5
B6
B7
B8
B11
B12
NDVI
NDWI
NDBI
LST
Elevation
```

You now have **14 features**, and many of them are correlated.

You want to reduce the dataset to perhaps **4–5 variables while retaining as much information as possible**.

### Which method would you choose?

**A. Logistic Regression**

**B. Random Forest**

**C. K-Means**

**D. PCA**

And most importantly:

**Why?**

